
# PSD parameter space: amplitude σ and timescale τ control burstiness

A 3×3 grid showing five stochastic-SFH realizations for each combination of
amplitude σ (vertical axis) and damping timescale τ (horizontal axis). Larger σ
produces more dramatic bursts; longer τ sustains those bursts. Each panel shows
the mean smooth SFH (dashed) and colored realizations. Bottom panels show
representative SEDs for σ alone (left) and τ alone (right), illustrating how
each parameter independently shapes the UV continuum and optical colors.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.plot import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

# --- Grid setup ---
n_grid = 256
log_age_grid = tengri.make_log_age_grid(n_grid)
d_log_age = float(log_age_grid[1] - log_age_grid[0])
t_lookback = 10.0**log_age_grid
t_gyr = np.array(t_lookback) / 1e9

mean_sfr = tengri.tsnorm(
    t_lookback, log_total_mass=10.0, peak_lbt=6e9, width=2e9, skew=0.5, trunc=3.0
)

# --- Parameter grid ---
sigmas = [0.2, 0.6, 1.2]
taus_myr = [30, 200, 1000]

fig, axes = plt.subplots(3, 3, figsize=(14, 12), sharex=True, sharey=True)
key_base = jax.random.PRNGKey(7)
mean_color = "0.30"
realiz_cmap = plt.get_cmap("viridis")

for i, sigma in enumerate(sigmas):
    for j, tau in enumerate(taus_myr):
        ax = axes[i, j]
        sqrt_p = tengri.compute_sqrt_power_drw(n_grid, d_log_age, sigma, tau * 1e6)

        # Plot 5 realizations with a consistent colormap
        n_realiz = 5
        for k in range(n_realiz):
            key = jax.random.fold_in(key_base, i * 100 + j * 10 + k)
            gp = tengri.generate_gp_fourier(key, sqrt_p, n_grid)
            variance = float(jnp.var(gp))
            sfr = mean_sfr * jnp.exp(gp - variance / 2.0)
            ax.plot(
                t_gyr,
                np.array(sfr),
                color=realiz_cmap(0.2 + 0.6 * k / max(n_realiz - 1, 1)),
                lw=0.9,
                alpha=0.8,
            )

        # Mean SFH on top so the trend is always visible.
        ax.plot(t_gyr, np.array(mean_sfr), color=mean_color, ls="--", lw=1.4, alpha=0.9)

        ax.set_xlim(0, 14)
        ax.set_yscale("log")
        # Mean peaks at ~5 Msun/yr; sigma=1.2 bursts reach ~30. Give breathing
        # room above and below so the trend reads clearly.
        ax.set_ylim(1e-1, 2e2)

        # Column titles only on the top row, row labels only on the left column.
        if j == 0:
            ax.set_ylabel(
                rf"$\sigma = {sigma}$" "\n" r"SFR [M$_\odot$/yr]",
                fontsize=11,
            )
        if i == 2:
            ax.set_xlabel("Lookback time [Gyr]")

fig.tight_layout(rect=[0, 0, 1, 0.96])

# --- Add bottom panels: σ and τ 1D sweeps in SED space ---
C_AA_PER_S = 2.998e18
ssp = tengri.load_ssp()

# Panel σ sweep (fixed τ)
model_sigma = tengri.SEDModel.build(
    ssp,
    sfh=[
        {"type": "const", "all_params": tengri.FIXED, "log_total_mass": 10.5},
        {
            "type": "field",
            "all_params": tengri.FIXED,
            "psd_sigma": tengri.Uniform(0.1, 3.5),
            "psd_tau_myr": 100.0,
        },
    ],
    dust={"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.2, "tau_bc": 0.3},
    redshift=tengri.Fixed(0.1),
)
baseline_sigma = dict(model_sigma.spec.sample(jax.random.PRNGKey(0)))

sigma_values = np.array([0.1, 0.5, 1.0, 2.0, 3.5])
norm_sigma = mpl.colors.Normalize(vmin=sigma_values.min(), vmax=sigma_values.max())
cmap = plt.get_cmap("viridis")

fig_bottom = plt.figure(figsize=(14, 4.5))
ax_sigma = fig_bottom.add_subplot(121)
ax_tau = fig_bottom.add_subplot(122)

key_base_sigma = jax.random.PRNGKey(0)
for i, sigma in enumerate(sigma_values):
    for k in range(3):
        params = {**baseline_sigma, "sfh_field_psd_sigma": jnp.float64(sigma)}
        key = jax.random.fold_in(key_base_sigma, i * 10 + k)
        out = model_sigma.predict(params)
        wave = np.asarray(model_sigma.wavelengths)
        nu = C_AA_PER_S / wave
        nu_l_nu = nu * np.asarray(out.rest_sed())
        ax_sigma.loglog(wave, nu_l_nu, color=cmap(norm_sigma(sigma)), lw=0.8, alpha=0.6)

ax_sigma.set_xlim(800, 3e4)
ax_sigma.set_ylim(1e40, 5e43)
ax_sigma.set_xlabel(r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]")
ax_sigma.set_ylabel(r"$\nu L_\nu$  [erg s$^{-1}$]")
cbar_sigma = fig_bottom.colorbar(
    plt.cm.ScalarMappable(norm=norm_sigma, cmap=cmap), ax=ax_sigma, pad=0.01
)
cbar_sigma.set_label(r"PSD amplitude $\sigma$")

# Panel τ sweep (fixed σ)
model_tau = tengri.SEDModel.build(
    ssp,
    sfh=[
        {"type": "const", "all_params": tengri.FIXED, "log_total_mass": 10.5},
        {
            "type": "field",
            "all_params": tengri.FIXED,
            "psd_sigma": 1.0,
            "psd_tau_myr": tengri.Uniform(30, 3000),
        },
    ],
    dust={"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.2, "tau_bc": 0.3},
    redshift=tengri.Fixed(0.1),
)
baseline_tau = dict(model_tau.spec.sample(jax.random.PRNGKey(0)))

tau_values = np.array([30, 100, 300, 1000, 3000])
norm_tau = mpl.colors.Normalize(vmin=tau_values.min(), vmax=tau_values.max())

key_base_tau = jax.random.PRNGKey(42)
for i, tau in enumerate(tau_values):
    for k in range(3):
        params = {**baseline_tau, "sfh_field_psd_tau_myr": jnp.float64(tau)}
        key = jax.random.fold_in(key_base_tau, i * 10 + k)
        out = model_tau.predict(params)
        wave = np.asarray(model_tau.wavelengths)
        nu = C_AA_PER_S / wave
        nu_l_nu = nu * np.asarray(out.rest_sed())
        ax_tau.loglog(wave, nu_l_nu, color=cmap(norm_tau(tau)), lw=0.8, alpha=0.6)

ax_tau.set_xlim(800, 3e4)
ax_tau.set_ylim(1e40, 5e43)
ax_tau.set_xlabel(r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]")
ax_tau.set_ylabel(r"$\nu L_\nu$  [erg s$^{-1}$]")
cbar_tau = fig_bottom.colorbar(
    plt.cm.ScalarMappable(norm=norm_tau, cmap=cmap), ax=ax_tau, pad=0.01
)
cbar_tau.set_label(r"PSD timescale $\tau$ [Myr]")

fig_bottom.tight_layout()
plt.savefig("plot_psd_burstiness.png", dpi=150, bbox_inches="tight")